In [8]:
import splitfolders
import os
import shutil
import tensorflow as tf
import random


# 1. Path Definitions and Global Variables

INPUT_ROOT_ORIGINAL = 'mri_data'          # Original dataset directory
OUTPUT_FOLDER = 'mri_data_split'         # Output directory for train/val/test
FLAT_INPUT_FOLDER = 'mri_data_flat_temp' # Temporary directory for flattened data

# Variables required for Cell 2
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
SPLIT_ROOT = OUTPUT_FOLDER
MODEL_PATH = 'final_best_model.keras'


# 2. Cleaning old folders, copying images, and flattening


# Remove old output & temp folders
if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)
if os.path.exists(FLAT_INPUT_FOLDER):
    shutil.rmtree(FLAT_INPUT_FOLDER)

os.makedirs(FLAT_INPUT_FOLDER, exist_ok=True)
print("Previous output folders removed and temporary folder created.")

print("Starting deep copy and flattening from subdirectories...")

image_count = 0
for class_name in os.listdir(INPUT_ROOT_ORIGINAL):
    original_class_path = os.path.join(INPUT_ROOT_ORIGINAL, class_name)
    flat_class_path = os.path.join(FLAT_INPUT_FOLDER, class_name)

    if not os.path.isdir(original_class_path):
        continue

    os.makedirs(flat_class_path, exist_ok=True)

    for dirpath, dirnames, filenames in os.walk(original_class_path):
        for filename in filenames:
            if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp')):
                src_file = os.path.join(dirpath, filename)
                dst_file = os.path.join(flat_class_path, filename)

                # Handle duplicated filenames
                if os.path.exists(dst_file):
                    base, ext = os.path.splitext(filename)
                    new_filename = f"{base}_{random.getrandbits(16)}{ext}"
                    dst_file = os.path.join(flat_class_path, new_filename)

                # Copy to ensure original data remains untouched
                shutil.copy(src_file, dst_file)
                image_count += 1

print(f" {image_count} images copied into temporary folder ({FLAT_INPUT_FOLDER}).")


# 3. Academic Splitting (Final Step)


print(" Starting academic split (80% / 10% / 10%)...")

splitfolders.ratio(
    FLAT_INPUT_FOLDER,
    output=OUTPUT_FOLDER,
    seed=42, #same split
    ratio=(0.8, 0.1, 0.1),
    group_prefix=None
)

print("\nSplit completed successfully. New folders are ready.")

# Remove temporary folder
shutil.rmtree(FLAT_INPUT_FOLDER)

Previous output folders removed and temporary folder created.
Starting deep copy and flattening from subdirectories...
 63325 images copied into temporary folder (mri_data_flat_temp).
 Starting academic split (80% / 10% / 10%)...


Copying files: 63325 files [00:32, 1974.00 files/s]



Split completed successfully. New folders are ready.


In [10]:
def load_data_from_split(subset_dir, shuffle_flag=False):

 # 1. Load the dataset from directory
    data = tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_ROOT, subset_dir),
        labels='inferred',
        label_mode='binary',
        image_size=IMAGE_SIZE,
        #same size
        interpolation='nearest',
        batch_size=BATCH_SIZE,
        shuffle=shuffle_flag,
    )

    # Save class names immediately to avoid class_names issues
    class_names = data.class_names

    # Function to normalize images and cast labels
    def normalize_and_cast(x, y):
        x = tf.cast(x, tf.float32) / 255.0
        y = tf.cast(y, tf.float32)
        return x, y

 # 2. Apply normalization and prefetch for performance
    data = data.map(normalize_and_cast, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)

    # Return dataset and class names
    return data, class_names


# Load the three datasets
train_dataset, class_names = load_data_from_split('train', shuffle_flag=True)
val_dataset, _ = load_data_from_split('val', shuffle_flag=False)
test_dataset, _ = load_data_from_split('test', shuffle_flag=False)  # Loaded but not used for training

print(f"Detected classes: {class_names}")
print("-" * 50)


Found 47807 files belonging to 2 classes.
Found 5975 files belonging to 2 classes.
Found 5978 files belonging to 2 classes.
Detected classes: ['Normal', 'Sick']
--------------------------------------------------


In [27]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


# Build the model and perform initial training (Feature Extraction)


# Load MobileNetV2 and freeze its layers
base_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze MobileNetV2 layers

# Add the classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

# Final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
initial_learning_rate = 0.0001
model.compile(
    optimizer=Adam(learning_rate=initial_learning_rate),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Callbacks: Save best model based on validation accuracy
model_checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True
)
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    mode='max',
    restore_best_weights=True
)

print("🏃 بدء التدريب الأولي (Feature Extraction)...")

# Train the model
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=[model_checkpoint, early_stopping]
)


Found 47807 files belonging to 2 classes.
Found 5975 files belonging to 2 classes.
Found 5978 files belonging to 2 classes.
تم اكتشاف الفئات: ['Normal', 'Sick']
--------------------------------------------------
🏃 بدء التدريب الأولي (Feature Extraction)...
Epoch 1/10
747/747 ━━━━━━━━━━━━━━━━━━━━ 165s 207ms/step - accuracy: 0.7023 - loss: 0.5781 - precision: 0.6549 - recall: 0.5797 - val_accuracy: 0.8854 - val_loss: 0.3128 - val_precision: 0.8746 - val_recall: 0.8398
Epoch 2/10
747/747 ━━━━━━━━━━━━━━━━━━━━ 111s 149ms/step - accuracy: 0.8718 - loss: 0.3190 - precision: 0.8587 - recall: 0.8216 - val_accuracy: 0.9245 - val_loss: 0.2278 - val_precision: 0.9216 - val_recall: 0.8910
Epoch 3/10
747/747 ━━━━━━━━━━━━━━━━━━━━ 108s 145ms/step - accuracy: 0.9106 - loss: 0.2416 - precision: 0.9021 - recall: 0.8766 - val_accuracy: 0.9374 - val_loss: 0.1830 - val_precision: 0.9434 - val_recall: 0.9009
Epoch 4/10
747/747 ━━━━━━━━━━━━━━━━━━━━ 150s 156ms/step - accuracy: 0.9305 - loss: 0.1973 - precision

In [28]:
#  Load the best saved model from the previous step
model = tf.keras.models.load_model("/content/final_best_model.keras")


# Unfreeze layers for Fine-Tuning

total_layers = len(model.layers)

# Unfreeze the last 24 layers (MobileNetV2 deeper layers + classification head)
fine_tune_at_index = total_layers - 24

# Make all layers trainable
for layer in model.layers:
    layer.trainable = True

# Freeze early layers that learned general features
for layer in model.layers[:fine_tune_at_index]:
    layer.trainable = False

num_unfrozen_layers = total_layers - fine_tune_at_index
print(f" تم فك تجميد آخر {num_unfrozen_layers} طبقة لعملية Fine-Tuning.")


# Recompile with very low learning rate

FINE_TUNE_EPOCHS = 10
initial_epochs = len(history.epoch)  # number of epochs already trained
total_epochs = initial_epochs + FINE_TUNE_EPOCHS

initial_learning_rate = 0.00001  # very small learning rate
optimizer = Adam(learning_rate=initial_learning_rate)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

print(f" تم إعادة تجميع النموذج بمعدل تعلم: {initial_learning_rate}")
print(f" بدء الضبط الدقيق (Fine-Tuning) لـ {FINE_TUNE_EPOCHS} دورات إضافية...")

# Reuse callbacks
history_fine_tune = model.fit(
    train_dataset,
    epochs=total_epochs,
    initial_epoch=initial_epochs,
    validation_data=val_dataset,
    callbacks=[
        ModelCheckpoint(filepath=MODEL_PATH, monitor='val_accuracy', mode='max', save_best_only=True),
        EarlyStopping(monitor='val_accuracy', patience=3, mode='max', restore_best_weights=True)
    ]
)

✅ تم فك تجميد آخر 24 طبقة لعملية Fine-Tuning.
✅ تم إعادة تجميع النموذج بمعدل تعلم: 1e-05
🏃 بدء الضبط الدقيق (Fine-Tuning) لـ 10 دورات إضافية...
Epoch 11/20
747/747 ━━━━━━━━━━━━━━━━━━━━ 139s 166ms/step - accuracy: 0.8994 - loss: 0.2475 - precision: 0.8834 - recall: 0.8686 - val_accuracy: 0.9625 - val_loss: 0.0922 - val_precision: 0.9530 - val_recall: 0.9553
Epoch 12/20
747/747 ━━━━━━━━━━━━━━━━━━━━ 107s 143ms/step - accuracy: 0.9590 - loss: 0.1101 - precision: 0.9541 - recall: 0.9452 - val_accuracy: 0.9704 - val_loss: 0.0761 - val_precision: 0.9732 - val_recall: 0.9537
Epoch 13/20
747/747 ━━━━━━━━━━━━━━━━━━━━ 142s 144ms/step - accuracy: 0.9708 - loss: 0.0780 - precision: 0.9655 - recall: 0.9631 - val_accuracy: 0.9731 - val_loss: 0.0620 - val_precision: 0.9715 - val_recall: 0.9623
Epoch 14/20
747/747 ━━━━━━━━━━━━━━━━━━━━ 109s 146ms/step - accuracy: 0.9795 - loss: 0.0574 - precision: 0.9770 - recall: 0.9728 - val_accuracy: 0.9769 - val_loss: 0.0538 - val_precision: 0.9832 - val_recall: 0.9

In [11]:
#Load the best final model (after both training phases)
final_model = tf.keras.models.load_model('../final_best_model.keras')

# Evaluate on the Test Set
print("\nStarting final evaluation on the separate test set...")

# Evaluate the model
loss, acc, precision, recall = final_model.evaluate(test_dataset)

print(f"\nFinal academic performance on the test set:")
print(f"   - Accuracy: {acc:.4f}")
print(f"   - Loss: {loss:.4f}")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall: {recall:.4f}")
print("-" * 50)


Starting final evaluation on the separate test set...


2025-11-30 02:34:05.676061: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


94/94 ━━━━━━━━━━━━━━━━━━━━ 22s 213ms/step - accuracy: 0.9851 - loss: 0.0664 - precision: 0.4091 - recall: 0.3813

Final academic performance on the test set:
   - Accuracy: 0.9726
   - Loss: 0.1358
   - Precision: 0.9935
   - Recall: 0.9390
--------------------------------------------------


### test new data

In [17]:
import h5py
import os
import numpy as np
from PIL import Image   # png image
import pandas as pd
from tqdm import tqdm


# Paths and output directories

INPUT_H5 = "/Users/saherelshewikh/PycharmProjects/Graduation_project/classification/ACDC_preprocessed/ACDC_testing_volumes"
OUTPUT_IMG = "ACDC_test_slices_fixed"
CSV_PATH = "ACDC_test_labels_fixed.csv"

os.makedirs(OUTPUT_IMG, exist_ok=True)


# Function to normalize image

def normalize_img(img):
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)


# Process each H5 file and save slices

data = []

for file in tqdm(sorted(os.listdir(INPUT_H5))):
    if file.endswith(".h5"):
        patient_id = file.split("_")[0]
        h5_path = os.path.join(INPUT_H5, file)

        with h5py.File(h5_path, "r") as f:
            vol = f['image'][:]       # shape: (num_slices, height, width)
            num_slices = vol.shape[0]

            for i in range(num_slices):
                slice_2d = vol[i, :, :]
                slice_2d = normalize_img(slice_2d)

                img_name = f"{patient_id}_slice{i}.png"
                save_path = os.path.join(OUTPUT_IMG, img_name)
                Image.fromarray(slice_2d).save(save_path)

                data.append([save_path, patient_id, "Unknown"])


# Save CSV file

df = pd.DataFrame(data, columns=["image", "patient", "label"])
df.to_csv(CSV_PATH, index=False)

print("✅ Done! PNG slices saved in:", OUTPUT_IMG)
print("✅ CSV saved at:", CSV_PATH)

100%|██████████| 100/100 [00:01<00:00, 50.28it/s]

✅ Done! PNG slices saved in: ACDC_test_slices_fixed
✅ CSV saved at: ACDC_test_labels_fixed.csv


In [13]:
import tensorflow as tf
import os
from tensorflow.keras.models import load_model


# 1. Paths and variables

EXTERNAL_DATA_ROOT = 'ACDC_test_slices_fixed'  # folder with Normal & Sick subfolders
MODEL_PATH = 'final_best_model.keras'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


# 2. Load model and external dataset


def normalize_and_cast(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    y = tf.cast(y, tf.float32)
    return x, y

try:
    final_model = load_model(MODEL_PATH)
    print("Successfully loaded the best trained model.")
except Exception as e:
    print(f" Error: Failed to load the model from {MODEL_PATH}. Message: {e}")
    raise SystemExit("Critical failure in loading model.")

print(f" Loading data from: {EXTERNAL_DATA_ROOT}...")

external_dataset = tf.keras.utils.image_dataset_from_directory(
    EXTERNAL_DATA_ROOT,
    labels='inferred',
    label_mode='binary',
    image_size=IMAGE_SIZE,
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_names=['Normal', 'Sick']  # Ensure Normal = 0, Sick = 1
)

external_class_names = external_dataset.class_names

external_dataset = external_dataset.map(normalize_and_cast, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)


# 3. Evaluate and display results


print("\n Starting final evaluation on the external independent dataset (ACDC)...")
print(f"   - Class 0: {external_class_names[0]}")
print(f"   - Class 1: {external_class_names[1]}")
print("-" * 50)

loss, acc, precision, recall = final_model.evaluate(external_dataset)

print(f"\n Final academic performance on external dataset:")
print(f"   - Accuracy: {acc:.4f}")
print(f"   - Loss: {loss:.4f}")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall: {recall:.4f}")

Successfully loaded the best trained model.
 Loading data from: ACDC_test_slices_fixed...
Found 140 files belonging to 2 classes.

 Starting final evaluation on the external independent dataset (ACDC)...
   - Class 0: Normal
   - Class 1: Sick
--------------------------------------------------
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 182ms/step - accuracy: 0.9411 - loss: 0.4030 - precision: 0.6359 - recall: 0.5702

 Final academic performance on external dataset:
   - Accuracy: 0.9000
   - Loss: 0.6535
   - Precision: 0.9667
   - Recall: 0.8286
